In [1]:
# Setup step: Install RabbitMQ server and start it
!apt-get update -qq && apt-get install -y -qq rabbitmq-server
!service rabbitmq-server start
!pip install -q pika
!mkdir -p stream

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package logrotate.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../00-logrotate_3.21.0-2build1_amd64.deb ...
Unpacking logrotate (3.21.0-2build1) ...
Selecting previously unselected package erlang-base.
Preparing to unpack .../01-erlang-base_1%3a25.3.2.8+dfsg-1ubuntu4.8_amd64.deb ...
Unpacking erlang-base (1:25.3.2.8+dfsg-1ubuntu4.8) ...
Selecting previously unselected package erlang-asn1.
Preparing to unpack .../02-erlang-asn1_1%3a25.3.2.8+dfsg-1ubuntu4.8_amd64.deb ...
Unpacking erlang-asn1 (1:25.3.2.8+dfsg-1ubuntu4.8) ...
Selecting previously unselected package erlang-crypto.
Preparing to unpack .../03-erlang-crypto_1%3a25.3.2.8+dfsg-1ubuntu4.8_amd64.deb ...
Unpacking erlang-crypto (1:25.3.2.8+dfsg-1ubuntu4.8) ...
Sele

In [2]:
%%writefile stream/dispatcher.py
import pika
import json
import time
import random
import uuid

cities = ['bangalore', 'chennai']
zones = ['koramangala', 'indiranagar', 'whitefield', 'tnagar']
types = ['ride', 'food']

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()
channel.exchange_declare(exchange='rapido-dispatch', exchange_type='topic')

print("Dispatcher started...")
message_count = 0
try:
    while True:
        message_count += 1
        city = random.choice(cities)
        zone = random.choice(zones)
        msg_type = random.choice(types)

        routing_key = f"{city}.{zone}.{msg_type}"
        payload = {
            "id": str(uuid.uuid4())[:8],
            "city": city,
            "type": msg_type,
            "ts": int(time.time())
        }

        # D3 Implementation: Omit "zone" on every 10th message
        if message_count % 10 != 0:
            payload["zone"] = zone
            print(f"[Dispatcher] Sent normal message #{message_count}: {routing_key}", flush=True)
        else:
            print(f"[Dispatcher] Sent BAD message #{message_count} (NO ZONE): {routing_key}", flush=True)

        channel.basic_publish(
            exchange='rapido-dispatch',
            routing_key=routing_key,
            body=json.dumps(payload)
        )
        time.sleep(1.5)
except KeyboardInterrupt:
    pass
finally:
    connection.close()

Writing stream/dispatcher.py


In [3]:
%%writefile stream/driver.py
import pika
import sys
import json

if len(sys.argv) < 3:
    print("Usage: python driver.py NAME PATTERN")
    sys.exit(1)

name = sys.argv[1]
pattern = sys.argv[2]

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()
channel.exchange_declare(exchange='rapido-dispatch', exchange_type='topic')

result = channel.queue_declare(queue='', exclusive=True)
queue_name = result.method.queue
channel.queue_bind(exchange='rapido-dispatch', queue=queue_name, routing_key=pattern)

print(f"[*] Driver {name} listening on pattern '{pattern}'...", flush=True)

def callback(ch, method, properties, body):
    try:
        data = json.loads(body.decode())
        if 'zone' not in data:
            print(f"[{name}] REJECTED message {data.get('id')} - missing zone! (NACK, requeue=False)", flush=True)
            ch.basic_nack(delivery_tag=method.delivery_tag, requeue=False)
        else:
            print(f"[{name}] PROCESSED message {data.get('id')} from zone {data['zone']}", flush=True)
            ch.basic_ack(delivery_tag=method.delivery_tag)
    except Exception as e:
        print(f"[{name}] Error processing message: {e}", flush=True)
        ch.basic_nack(delivery_tag=method.delivery_tag, requeue=False)

channel.basic_consume(queue=queue_name, on_message_callback=callback, auto_ack=False)

try:
    channel.start_consuming()
except KeyboardInterrupt:
    pass
finally:
    connection.close()

Writing stream/driver.py


In [4]:
# Executing the simulation concurrently with at least three drivers (Ravi, Arjun, and Control)
import subprocess
import time
import os

print("--- Launching dedicated Drivers and Dispatcher concurrently ---", flush=True)

# Launching 3 drivers as required
p_ravi = subprocess.Popen(['python', '-u', 'stream/driver.py', 'Ravi', 'bangalore.koramangala.*'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
p_arjun = subprocess.Popen(['python', '-u', 'stream/driver.py', 'Arjun', 'bangalore.*.ride'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
p_control = subprocess.Popen(['python', '-u', 'stream/driver.py', 'Control', 'bangalore.#'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(2)

p_dispatch = subprocess.Popen(['python', '-u', 'stream/dispatcher.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

processes = [("Ravi", p_ravi), ("Arjun", p_arjun), ("Control", p_control), ("Dispatcher", p_dispatch)]

for name, p in processes:
    os.set_blocking(p.stdout.fileno(), False)

try:
    # Running for 20 seconds to show routing and bad message NACK behavior across all 3 drivers
    end_time = time.time() + 20
    while time.time() < end_time:
        for name, p in processes:
            line = p.stdout.readline()
            if line:
                print(line.strip(), flush=True)
        time.sleep(0.1)
finally:
    print("\n--- Shutting down processes ---", flush=True)
    p_dispatch.terminate()
    p_ravi.terminate()
    p_arjun.terminate()
    p_control.terminate()

--- Launching dedicated Drivers and Dispatcher concurrently ---
[*] Driver Ravi listening on pattern 'bangalore.koramangala.*'...
[*] Driver Arjun listening on pattern 'bangalore.*.ride'...
[*] Driver Control listening on pattern 'bangalore.#'...
[Control] PROCESSED message 4abcc2fc from zone whitefield
Dispatcher started...
[Dispatcher] Sent normal message #1: bangalore.whitefield.food
[Control] PROCESSED message c50d1861 from zone indiranagar
[Dispatcher] Sent normal message #2: bangalore.indiranagar.food
[Arjun] PROCESSED message a8a49470 from zone whitefield
[Control] PROCESSED message a8a49470 from zone whitefield
[Dispatcher] Sent normal message #3: bangalore.whitefield.ride
[Arjun] PROCESSED message 02bb6558 from zone indiranagar
[Control] PROCESSED message 02bb6558 from zone indiranagar
[Dispatcher] Sent normal message #4: bangalore.indiranagar.ride
[Dispatcher] Sent normal message #5: chennai.koramangala.ride
[Dispatcher] Sent normal message #6: chennai.koramangala.ride
[Contr

In [5]:
# Re-write stream/driver.py to reject messages without 'zone'
driver_d3_code = """
import pika
import sys
import json

if len(sys.argv) < 3:
    print("Usage: python driver.py NAME PATTERN")
    sys.exit(1)

name = sys.argv[1]
pattern = sys.argv[2]

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()
channel.exchange_declare(exchange='rapido-dispatch', exchange_type='topic')

result = channel.queue_declare(queue='', exclusive=True)
queue_name = result.method.queue
channel.queue_bind(exchange='rapido-dispatch', queue=queue_name, routing_key=pattern)

print(f"[*] Driver {name} listening on pattern '{pattern}'...", flush=True)

def callback(ch, method, properties, body):
    try:
        data = json.loads(body.decode())
        if 'zone' not in data:
            print(f"[{name}] REJECTED message {data.get('id')} - missing zone! (NACK, requeue=False)", flush=True)
            ch.basic_nack(delivery_tag=method.delivery_tag, requeue=False)
        else:
            print(f"[{name}] PROCESSED message {data.get('id')} from zone {data['zone']}", flush=True)
            ch.basic_ack(delivery_tag=method.delivery_tag)
    except Exception as e:
        print(f"[{name}] Error processing message: {e}", flush=True)
        ch.basic_nack(delivery_tag=method.delivery_tag, requeue=False)

channel.basic_consume(queue=queue_name, on_message_callback=callback, auto_ack=False)

try:
    channel.start_consuming()
except KeyboardInterrupt:
    pass
finally:
    connection.close()
"""

with open('stream/driver.py', 'w') as f:
    f.write(driver_d3_code.strip())


In [6]:
%%writefile stream/dispatcher.py
import pika
import json
import time
import random
import uuid

cities = ['bangalore', 'chennai']
zones = ['koramangala', 'indiranagar', 'whitefield', 'tnagar']
types = ['ride', 'food']

connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
channel = connection.channel()
channel.exchange_declare(exchange='rapido-dispatch', exchange_type='topic')

print("Dispatcher started...")
message_count = 0
try:
    while True:
        message_count += 1
        city = random.choice(cities)
        zone = random.choice(zones)
        msg_type = random.choice(types)

        # Force Bangalore for bad messages so they are received by our active drivers
        if message_count % 10 == 0:
            city = 'bangalore'

        routing_key = f"{city}.{zone}.{msg_type}"
        payload = {
            "id": str(uuid.uuid4())[:8],
            "city": city,
            "type": msg_type,
            "ts": int(time.time())
        }

        # D3 Implementation: Omit "zone" on every 10th message
        if message_count % 10 != 0:
            payload["zone"] = zone
            print(f"[Dispatcher] Sent normal message #{message_count}: {routing_key}", flush=True)
        else:
            print(f"[Dispatcher] Sent BAD message #{message_count} (NO ZONE): {routing_key}", flush=True)

        channel.basic_publish(
            exchange='rapido-dispatch',
            routing_key=routing_key,
            body=json.dumps(payload)
        )
        time.sleep(1.5)
except KeyboardInterrupt:
    pass
finally:
    connection.close()

Overwriting stream/dispatcher.py


In [7]:
# Executing the simulation concurrently with fixed dispatcher
import subprocess
import time
import os

print("--- Launching dedicated Drivers and Fixed Dispatcher concurrently ---", flush=True)

p_ravi = subprocess.Popen(['python', '-u', 'stream/driver.py', 'Ravi', 'bangalore.koramangala.*'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
p_arjun = subprocess.Popen(['python', '-u', 'stream/driver.py', 'Arjun', 'bangalore.*.ride'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
p_control = subprocess.Popen(['python', '-u', 'stream/driver.py', 'Control', 'bangalore.#'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(2)

p_dispatch = subprocess.Popen(['python', '-u', 'stream/dispatcher.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

processes = [("Ravi", p_ravi), ("Arjun", p_arjun), ("Control", p_control), ("Dispatcher", p_dispatch)]

for name, p in processes:
    os.set_blocking(p.stdout.fileno(), False)

try:
    # Run for 20 seconds to guarantee we hit message #10 and see the REJECTED log output
    end_time = time.time() + 20
    while time.time() < end_time:
        for name, p in processes:
            line = p.stdout.readline()
            if line:
                print(line.strip(), flush=True)
        time.sleep(0.1)
finally:
    print("\n--- Shutting down processes ---", flush=True)
    p_dispatch.terminate()
    p_ravi.terminate()
    p_arjun.terminate()
    p_control.terminate()

--- Launching dedicated Drivers and Fixed Dispatcher concurrently ---
[*] Driver Ravi listening on pattern 'bangalore.koramangala.*'...
[*] Driver Arjun listening on pattern 'bangalore.*.ride'...
[*] Driver Control listening on pattern 'bangalore.#'...
[Control] PROCESSED message ba021807 from zone indiranagar
Dispatcher started...
[Dispatcher] Sent normal message #1: bangalore.indiranagar.food
[Dispatcher] Sent normal message #2: chennai.tnagar.ride
[Dispatcher] Sent normal message #3: chennai.whitefield.ride
[Arjun] PROCESSED message d804ecb2 from zone tnagar
[Control] PROCESSED message d804ecb2 from zone tnagar
[Dispatcher] Sent normal message #4: bangalore.tnagar.ride
[Dispatcher] Sent normal message #5: chennai.whitefield.food
[Control] PROCESSED message 6301ac17 from zone tnagar
[Dispatcher] Sent normal message #6: bangalore.tnagar.food
[Control] PROCESSED message cbdda37c from zone tnagar
[Dispatcher] Sent normal message #7: bangalore.tnagar.food
[Dispatcher] Sent normal message

In [8]:
import shutil
from google.colab import files

# Zip the 'stream' folder
shutil.make_archive('stream', 'zip', 'stream')

# Download the zip file
files.download('stream.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>